# Oráculo do ScribblePrompt — qual é o teto da rede final neste pipeline?

**Pergunta:** se a MarkerUNet fosse *perfeita*, até onde o ScribblePrompt congelado conseguiria chegar? Aqui **não há
treino**: entregamos ao ScribblePrompt marcadores tirados do próprio ground truth ("oráculos") e medimos o Dice.

Por que isso importa ([08-pendencias.md](../../docs/estudo/08-pendencias.md)):

- **PD-01:** o melhor experimento (exp. 4, Dice 0,648) fica abaixo do Cellpose sozinho (Dice 0,810).
- **PD-03:** este é o experimento E3 da investigação do exp. 1, que nunca foi feito. Se nem com o marcador ideal o
  ScribblePrompt passar do Cellpose, nenhuma MarkerUNet vai passar, e o problema está na rede final, não no treino.
- **PD-04:** hoje o canal negativo é o complemento denso do marcador. Foi decidido usar **só scribbles positivos**; aqui as
  duas formas são comparadas.
- **PD-05:** o ScribblePrompt foi treinado em 128×128, mas é totalmente convolucional, e o artigo diz que o método "não é
  preso a uma resolução" (também foi avaliado em 256²). Aqui a resolução de entrada da rede é variada.

**Protocolo:** todas as combinações de marcador × canal negativo × resolução, nas 30 imagens de treino e nas 14 de teste,
com as mesmas métricas dos notebooks de experimento (limiar 0,5, média por imagem). A comparação que decide é a do
**treino** (PD-02); o teste aparece só como referência.

> Um teste anterior com **antialias** na redução para 128² (PD-37) piorou os resultados: os scribbles deixam de ser 0/1
> e viram um "borrão" fora da distribuição de treino. Por isso o antialias ficou fora deste notebook (detalhes na PD-37).

## 0. Setup

**Colab:** rode as duas células abaixo (clona o repositório e instala o `requirements.txt`). GPU ajuda, mas não é
necessária: o ScribblePrompt-UNet trabalha em 128×128 e roda bem em CPU.

In [ ]:
# Encontra a raiz do repositório (local ou Colab) e a coloca no sys.path — mesmo padrão dos notebooks de experimento.
import os
import sys

repo_dir_name = "cell-fuzzy-seg"


def find_repo_root(start="."):
    p = os.path.abspath(start)
    prev = None
    while p != prev:
        if os.path.isdir(os.path.join(p, "src")) or os.path.isdir(os.path.join(p, ".git")):
            return p
        prev = p
        p = os.path.dirname(p)
    return None


if "google.colab" in sys.modules:
    base = "/content"
    repo_root = find_repo_root(base) or os.path.join(base, repo_dir_name)
    if not os.path.exists(repo_root):
        get_ipython().system(f"git clone --branch homolog https://github.com/Pedro-io/cell-fuzzy-seg.git {repo_root}")
else:
    repo_root = find_repo_root(os.getcwd()) or os.getcwd()

os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("CWD:", os.getcwd())

In [ ]:
import sys

if "google.colab" in sys.modules:
    get_ipython().system("python -m pip install -r requirements.txt -q")
else:
    print("Ambiente local: verifique se as dependências estão instaladas.")

## 1. Imports e configuração

In [ ]:
import hashlib
import json
import os

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import ndimage

from src.models.networks.final_segmentation.scribble_prompting_network import ScribblePromptingNetwork

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PREPROCESSED_DIR = os.path.join("data_source", "MoNuSegPreprocessed")
CKPT_DIR = os.environ.get("SP_CKPT_DIR", "checkpoints")   # onde o checkpoint do ScribblePrompt é salvo
THRESHOLD = 0.5                                            # o mesmo limiar dos notebooks de experimento
SPLITS = ["train", "test"]
RESOLUCOES = [(128, 128), (256, 256), (512, 512)]          # entrada do ScribblePrompt (divisível por 16); 128 é a atual
print("Device:", DEVICE)

## 2. Dados pré-processados

Os mesmos `.npy` que os experimentos usam. `rgba[..., 3]` é a máscara binária do Cellpose.

In [ ]:
def load_split(split):
    base = os.path.join(PREPROCESSED_DIR, split)
    ids = sorted(f[:-4] for f in os.listdir(os.path.join(base, "image")) if f.endswith(".npy"))
    return [
        {
            "id": i,
            "image": np.load(os.path.join(base, "image", f"{i}.npy")),                 # (H, W, 3) uint8
            "rgba": np.load(os.path.join(base, "rgba", f"{i}.npy")),                   # (H, W, 4) float32
            "ground_truth": np.load(os.path.join(base, "ground_truth", f"{i}.npy")),   # (H, W) uint8 0/1
        }
        for i in ids
    ]


data = {split: load_split(split) for split in SPLITS}
print({split: len(samples) for split, samples in data.items()})

## 3. Rede final congelada

A mesma classe usada no treino. O SHA-256 do checkpoint é impresso para ser registrado (PD-15: hoje o download não confere
hash).

In [ ]:
os.makedirs(CKPT_DIR, exist_ok=True)
ckpt = ScribblePromptingNetwork.download_checkpoint(CKPT_DIR)
with open(ckpt, "rb") as f:
    print("SHA-256 do checkpoint:", hashlib.sha256(f.read()).hexdigest())

net = ScribblePromptingNetwork(
    checkpoint=ckpt,
    device=DEVICE,
    resize_output=True,
    scribble_mode="sharpened",
    scribble_temperature=10.0,
)

## 4. Marcadores-oráculo (derivados do GT)

Todos binários, em 1000×1000. "Componente" = região conexa do GT binário. Como núcleos colados se fundem (PD-07), um
componente pode ter vários núcleos.

| Marcador | O que é | Para que serve |
|---|---|---|
| `gt` | o GT inteiro | teto "trivial": o ScribblePrompt consegue ao menos devolver o que recebeu? |
| `miolo` | a metade mais interna de cada componente (EDT normalizada **por componente** ≥ 0,5) | marcador "bom" no sentido da tese: menor e longe da borda |
| `centros` | um disco de raio 4 px no ponto mais interno de cada componente | marcador mínimo, como cliques no centro |
| `cellpose` | a máscara do Cellpose | o que uma MarkerUNet que só copiasse a entrada produziria |
| `vazio` | nenhum marcador | sanidade: o que o ScribblePrompt faz sem prompt |

In [ ]:
def _componentes(gt):
    fg = gt > 0
    lab, n = ndimage.label(fg)
    dt = ndimage.distance_transform_edt(fg)
    return fg, lab, n, dt


def marcador_gt(sample):
    return (sample["ground_truth"] > 0).astype(np.float32)


def marcador_miolo(sample, frac=0.5):
    fg, lab, n, dt = _componentes(sample["ground_truth"])
    if n == 0:
        return np.zeros(fg.shape, np.float32)
    maximos = ndimage.maximum(dt, lab, index=np.arange(1, n + 1))
    maximos = np.concatenate([[1.0], np.maximum(maximos, 1e-6)])   # índice 0 = fundo
    return (fg & (dt / maximos[lab] >= frac)).astype(np.float32)


def marcador_centros(sample, raio=4):
    fg, lab, n, dt = _componentes(sample["ground_truth"])
    out = np.zeros(fg.shape, np.uint8)
    for y, x in ndimage.maximum_position(dt, lab, index=np.arange(1, n + 1)):
        cv2.circle(out, (int(x), int(y)), raio, 1, thickness=-1)
    return out.astype(np.float32)


def marcador_cellpose(sample):
    return (sample["rgba"][..., 3] > 0).astype(np.float32)


def marcador_vazio(sample):
    return np.zeros(sample["ground_truth"].shape, np.float32)


MARCADORES = {
    "gt": marcador_gt,
    "miolo": marcador_miolo,
    "centros": marcador_centros,
    "cellpose": marcador_cellpose,
    "vazio": marcador_vazio,
}

## 5. Do marcador à segmentação

- `negativo="atual"`: o marcador de 1 canal vai para o wrapper, que aplica `pos = σ(10(m−0,5))` e
  `neg = σ(10(0,5−m))`. É o que o pipeline faz hoje: todo pixel não marcado vira "fundo".
- `negativo="so_positivo"`: `[m, 0]`. É a decisão tomada para a PD-04.
- `resolucao`: o tamanho para o qual o wrapper reduz imagem e scribbles antes da rede (`net.input_size`). A saída volta a
  1000² por bilinear, exatamente como no pipeline.

In [ ]:
def segmentar(sample, marcador, negativo, resolucao):
    image = torch.from_numpy(sample["image"].transpose(2, 0, 1).copy()).float()[None]   # (1, 3, H, W) em [0, 255]
    m = torch.from_numpy(marcador)[None, None]                                         # (1, 1, H, W)

    if negativo == "atual":
        scribbles = m                                          # o wrapper aplica o sharpened (T=10)
    elif negativo == "so_positivo":
        scribbles = torch.cat([m, torch.zeros_like(m)], dim=1)
    else:
        raise ValueError(negativo)

    net.input_size = tuple(resolucao)                          # o wrapper usa este tamanho em rescale_inputs
    prob = net.predict({"image": image, "scribbles": scribbles})
    return prob[0, 0].cpu().numpy()


def metricas(prob, gt, threshold=THRESHOLD):
    # Mesmas fórmulas de compute_binary_metrics dos notebooks de experimento.
    p = prob >= threshold
    g = gt > 0
    tp = float((p & g).sum())
    fp = float((p & ~g).sum())
    fn = float((~p & g).sum())
    eps = 1e-8
    return {
        "dice": 2 * tp / (2 * tp + fp + fn + eps),
        "iou": tp / (tp + fp + fn + eps),
        "precision": tp / (tp + fp + eps),
        "recall": tp / (tp + fn + eps),
        "mass_ratio": float(p.sum()) / max(float(g.sum()), 1.0),
    }

## 6. Rodar todas as combinações

In [ ]:
linhas = []
for split in SPLITS:
    for sample in data[split]:
        gt = sample["ground_truth"]
        base = metricas(marcador_cellpose(sample), gt)
        linhas.append({"split": split, "id": sample["id"], "marcador": "(Cellpose sozinho)",
                       "negativo": "-", "resolucao": "-", "tam_rel_marcador": np.nan, **base})
        for nome, fn in MARCADORES.items():
            marcador = fn(sample)
            tam_rel = float(marcador.sum()) / max(float((gt > 0).sum()), 1.0)
            for negativo in ["atual", "so_positivo"]:
                for resolucao in RESOLUCOES:
                    with torch.no_grad():
                        prob = segmentar(sample, marcador, negativo, resolucao)
                    linhas.append({"split": split, "id": sample["id"], "marcador": nome, "negativo": negativo,
                                   "resolucao": resolucao[0], "tam_rel_marcador": tam_rel, **metricas(prob, gt)})
    print(f"{split}: ok")

df = pd.DataFrame(linhas)
print(len(df), "linhas")

## 7. Resultados

`tam_rel_marcador` = área do marcador / área do GT (é o que o L_size da tese mede).

In [ ]:
pd.set_option("display.width", 200)
resumo = (
    df.groupby(["split", "marcador", "negativo", "resolucao"], sort=False)[
        ["dice", "iou", "precision", "recall", "mass_ratio", "tam_rel_marcador"]
    ]
    .mean()
    .round(3)
)
print(resumo.to_string())

In [ ]:
# Visão compacta: Dice médio no TREINO (a comparação que decide) — linhas = marcador, colunas = resolução × negativo.
tab = (
    df[(df.split == "train") & (df.marcador != "(Cellpose sozinho)")]
    .pivot_table(index="marcador", columns=["resolucao", "negativo"], values="dice", aggfunc="mean", sort=False)
    .round(3)
)
cellpose_train = df[(df.split == "train") & (df.marcador == "(Cellpose sozinho)")].dice.mean()
print(tab.to_string())
print(f"\nCellpose sozinho (treino): Dice {cellpose_train:.3f}")

## 8. Como ler

- **Algum oráculo passa do Cellpose no treino?** Se `gt`/`miolo` com `so_positivo` passarem de ~0,80, o ScribblePrompt
  *pode* superar o Cellpose, e vale continuar ajustando a MarkerUNet e as losses. Se nenhum passar, o teto da rede final
  fica abaixo da linha de base, e é preciso outra rede final ou outra forma de entregar o marcador.
- **`atual` × `so_positivo`:** mede o custo do canal negativo denso (PD-04). Com `atual`, a saída tende a ser o próprio
  marcador (precisão ~1, recall ≈ tamanho do marcador); com `so_positivo`, a rede "cresce" a partir do marcador.
- **`resolucao`:** mede quanto a entrada de 128² limita o resultado (PD-05).
- **`cellpose` como marcador:** se o ScribblePrompt piorar a máscara do Cellpose, uma MarkerUNet que "só copie" a entrada
  já parte abaixo da linha de base.
- **`vazio`:** confirma que o prompt é o que guia a saída.

In [ ]:
# Visualização: recorte de uma imagem de teste, oráculos com negativo "so_positivo" em 256².
import matplotlib.pyplot as plt

sample = data["test"][0]
y0, x0, lado = 300, 300, 250
fatia = (slice(y0, y0 + lado), slice(x0, x0 + lado))
nomes = ["gt", "miolo", "centros", "cellpose"]
fig, axes = plt.subplots(2, len(nomes) + 1, figsize=(4 * (len(nomes) + 1), 8))
axes[0, 0].imshow(sample["image"][fatia]); axes[0, 0].set_title(sample["id"])
axes[1, 0].imshow(sample["ground_truth"][fatia], cmap="gray"); axes[1, 0].set_title("ground truth")
for j, nome in enumerate(nomes, start=1):
    marcador = MARCADORES[nome](sample)
    with torch.no_grad():
        prob = segmentar(sample, marcador, "so_positivo", (256, 256))
    d = metricas(prob, sample["ground_truth"])["dice"]
    axes[0, j].imshow(marcador[fatia], cmap="gray"); axes[0, j].set_title(f"marcador: {nome}")
    axes[1, j].imshow(prob[fatia] >= THRESHOLD, cmap="gray"); axes[1, j].set_title(f"saída (Dice da imagem {d:.3f})")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()